In [1]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [13]:
#tweet loading
tweets = pd.read_csv("file:///home/chakuunaa/BD_ADA_CA2sem2/data/stocktweet.csv")

tweets.head()

,id,date,ticker,tweet
0,100001,01/01/2020,AMZN,$AMZN Dow futures up by 100 points already 🥳
1,100002,01/01/2020,TSLA,$TSLA Daddy's drinkin' eArly tonight! Here's t...
2,100003,01/01/2020,AAPL,$AAPL We’ll been riding since last December fr...
3,100004,01/01/2020,TSLA,"$TSLA happy new year, 2020, everyone🍷🎉🙏"
4,100005,01/01/2020,TSLA,"$TSLA haha just a collection of greats...""Mars..."


In [14]:
tweets.shape

(10000, 4)

In [15]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [16]:
tweets["clean_tweet"] = tweets["tweet"].apply(clean_tweet)

tweets[["tweet", "clean_tweet"]].head()

,tweet,clean_tweet
0,$AMZN Dow futures up by 100 points already 🥳,amzn dow futures up by 100 points already
1,$TSLA Daddy's drinkin' eArly tonight! Here's t...,tsla daddys drinkin early tonight heres to a p...
2,$AAPL We’ll been riding since last December fr...,aapl well been riding since last december from...
3,"$TSLA happy new year, 2020, everyone🍷🎉🙏",tsla happy new year 2020 everyone
4,"$TSLA haha just a collection of greats...""Mars...",tsla haha just a collection of greatsmars rofl...


In [17]:
analyzer = SentimentIntensityAnalyzer()

In [18]:
tweets["sentiment_score"] = tweets["clean_tweet"].apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)

tweets[["ticker", "date", "clean_tweet", "sentiment_score"]].head()

,ticker,date,clean_tweet,sentiment_score
0,AMZN,01/01/2020,amzn dow futures up by 100 points already,0.0000
1,TSLA,01/01/2020,tsla daddys drinkin early tonight heres to a p...,0.0000
2,AAPL,01/01/2020,aapl well been riding since last december from...,0.2732
3,TSLA,01/01/2020,tsla happy new year 2020 everyone,0.5719
4,TSLA,01/01/2020,tsla haha just a collection of greatsmars rofl...,0.7717


In [19]:

def sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

tweets["sentiment_label"] = tweets["sentiment_score"].apply(sentiment_label)

tweets[["sentiment_score", "sentiment_label"]].head()

,sentiment_score,sentiment_label
0,0.0000,neutral
1,0.0000,neutral
2,0.2732,positive
3,0.5719,positive
4,0.7717,positive


In [20]:
daily_sentiment = tweets.groupby(["date", "ticker"]).agg(
    avg_sentiment=("sentiment_score", "mean"),
    tweet_count=("sentiment_score", "count")
).reset_index()

daily_sentiment.head()

,date,ticker,avg_sentiment,tweet_count
0,01/01/2020,AAPL,0.273200,1
1,01/01/2020,AMZN,0.000000,1
2,01/01/2020,TSLA,0.111975,4
3,01/02/2020,AAPL,0.261133,3
4,01/02/2020,AMZN,0.549900,1


In [21]:
daily_sentiment.groupby("ticker")["avg_sentiment"].mean().sort_values(ascending=False).head(10)

ticker
MA      0.327625
PYPL    0.227606
MCD     0.203543
HD      0.195737
SBUX    0.187796
NKE     0.184058
JNJ     0.167395
AMZN    0.155989
XOM     0.149006
V       0.145305
Name: avg_sentiment, dtype: float64

In [25]:
daily_sentiment.to_csv(
    "/home/chakuunaa/BD_ADA_CA2sem2/data/daily_sentiment.csv",
    index=False
)

tweets.to_csv(
    "/home/chakuunaa/BD_ADA_CA2sem2/data/tweets_with_sentiment.csv",
    index=False
)

In [26]:
import os

os.listdir("/home/chakuunaa/BD_ADA_CA2sem2/data")[:20]

['stockprice',
 'daily_sentiment.csv',
 'stocktweet.csv',
 'processed_tweet_counts',
 'tweets_with_sentiment.csv']